<a href="https://colab.research.google.com/github/hinamehboobcs/Industry-Project/blob/main/VanilaDQN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [46]:
import os

# ======================================================
# Project Root
# ======================================================

PROJECT_ROOT = "/content/drive/MyDrive/VanillaDQN"

# ======================================================
# Folder Structure
# ======================================================

folders = [
    "config",
    "data",
    "utils",
    "constraints",
    "rewards",
    "environment",
    "agents",
    "training",
    "evaluation",
    "visualization",
    "saved_models",
    "results",
    "notebooks"
]

# ======================================================
# Python Files
# ======================================================

python_files = {
    "config": [
        "__init__.py",
        "settings.py"
    ],

    "utils": [
        "__init__.py",
        "preprocessing.py",
        "distance.py",
        "helpers.py"
    ],

    "constraints": [
        "__init__.py",
        "constraint_engine.py"
    ],

    "rewards": [
        "__init__.py",
        "reward_function.py"
    ],

    "environment": [
        "__init__.py",
        "healthcare_env.py"
    ],

    "agents": [
        "__init__.py",
        "replay_buffer.py",
        "dqn_network.py",
        "dqn_agent.py"
    ],

    "training": [
        "__init__.py",
        "train.py"
    ],

    "evaluation": [
        "__init__.py",
        "evaluate.py",
        "metrics.py"
    ],

    "visualization": [
        "__init__.py",
        "plots.py"
    ]
}

# ======================================================
# Create Root Folder
# ======================================================

os.makedirs(PROJECT_ROOT, exist_ok=True)

# ======================================================
# Create Folders
# ======================================================

for folder in folders:

    os.makedirs(
        os.path.join(PROJECT_ROOT, folder),
        exist_ok=True
    )

# ======================================================
# Create Python Files
# ======================================================

for folder, files in python_files.items():

    folder_path = os.path.join(
        PROJECT_ROOT,
        folder
    )

    for file in files:

        file_path = os.path.join(
            folder_path,
            file
        )

        if not os.path.exists(file_path):

            with open(file_path, "w") as f:

                f.write("")

# ======================================================
# Other Files
# ======================================================

other_files = [
    "README.md",
    "requirements.txt"
]

for file in other_files:

    file_path = os.path.join(
        PROJECT_ROOT,
        file
    )

    if not os.path.exists(file_path):

        with open(file_path, "w") as f:

            f.write("")

print("=" * 60)
print("VanillaDQN Project Created Successfully!")
print("=" * 60)

for root, dirs, files in os.walk(PROJECT_ROOT):

    level = root.replace(PROJECT_ROOT, "").count(os.sep)

    indent = " " * 4 * level

    print(f"{indent}{os.path.basename(root)}/")

    sub_indent = " " * 4 * (level + 1)

    for file in files:

        print(f"{sub_indent}{file}")

# Define the content for utils/preprocessing.py
preprocessing_code = '''
import pandas as pd
import numpy as np
import os
from ast import literal_eval # To convert string representations of lists/sets to actual lists/sets

PROJECT_ROOT = "/content/drive/MyDrive/VanillaDQN"
DATA_PATH = os.path.join(PROJECT_ROOT, "data")

def load_carers():
    carers_path = os.path.join(DATA_PATH, "carers_available.csv") # Corrected filename
    carers = pd.read_csv(carers_path)
    return carers

def load_visits():
    visits_path = os.path.join(DATA_PATH, "visits.csv")
    visits = pd.read_csv(visits_path)
    return visits

def preprocess_data():
    carers = load_carers()
    visits = load_visits()

    # Convert string representation of sets to actual sets
    for col in ["Skills", "Languages"]:
        if col in carers.columns:
            carers[col] = carers[col].apply(lambda x: literal_eval(x) if isinstance(x, str) else x)

    for col in ["RequiredSkills", "PreferredLanguage"]:
        if col in visits.columns:
            visits[col] = visits[col].apply(lambda x: literal_eval(x) if isinstance(x, str) else x)

    # Encode ordinal categorical features in visits
    if "PriorityLevel" in visits.columns:
        priority_mapping = {'Low': 0, 'Medium': 1, 'High': 2}
        visits['PriorityLevel'] = visits['PriorityLevel'].map(priority_mapping).fillna(-1).astype(int) # -1 for unknown

    if "ComplexityLevel" in visits.columns:
        complexity_mapping = {'Low': 0, 'Medium': 1, 'High': 2}
        visits['ComplexityLevel'] = visits['ComplexityLevel'].map(complexity_mapping).fillna(-1).astype(int) # -1 for unknown

    # Convert binary categorical features
    if "TwoCarerRequired" in visits.columns:
        visits['TwoCarerRequired'] = visits['TwoCarerRequired'].map({'No': 0, 'Yes': 1}).fillna(-1).astype(int)
    if "ContinuityRequirement" in visits.columns: # Assuming similar binary structure
        visits['ContinuityRequirement'] = visits['ContinuityRequirement'].map({'No': 0, 'Yes': 1}).fillna(-1).astype(int)

    # Encode nominal categorical features (label encoding)
    # For Gender (assuming Carers only)
    if "Gender" in carers.columns:
        carers['Gender'] = carers['Gender'].map({'Male': 0, 'Female': 1, 'Other': 2}).fillna(-1).astype(int)

    # For WorkerType (assuming Carers only)
    if "WorkerType" in carers.columns:
        carers['WorkerType'] = carers['WorkerType'].astype('category').cat.codes

    # For VehicleType (assuming Carers only)
    if "VehicleType" in carers.columns:
        carers['VehicleType'] = carers['VehicleType'].astype('category').cat.codes

    # For PreferredGender in visits
    if "PreferredGender" in visits.columns:
        visits['PreferredGender'] = visits['PreferredGender'].map({'Male': 0, 'Female': 1, 'Any': 2}).fillna(-1).astype(int)


    # Convert Day of Week to numerical
    day_mapping = {'Monday': 0, 'Tuesday': 1, 'Wednesday': 2, 'Thursday': 3, 'Friday': 4, 'Saturday': 5, 'Sunday': 6}
    if "DayOfWeek" in carers.columns:
        carers['DayOfWeek'] = carers['DayOfWeek'].map(day_mapping).fillna(-1).astype(int)
    if "Day" in visits.columns:
        visits['Day'] = visits['Day'].map(day_mapping).fillna(-1).astype(int)

    # Convert time strings to minutes from midnight
    def time_to_minutes(time_str):
        if pd.isnull(time_str) or time_str == '':
            return np.nan
        if isinstance(time_str, (int, float)): # Already numerical
            return time_str
        parts = str(time_str).split(':')
        if len(parts) == 2:
            try:
                return int(parts[0]) * 60 + int(parts[1])
            except ValueError: # Handle cases like '24:00' or other invalid time strings
                return np.nan
        return np.nan

    for col in ["ShiftStart", "ShiftEnd"]:
        if col in carers.columns:
            carers[col] = carers[col].apply(time_to_minutes)

    for col in ["TimeWindowStart", "TimeWindowEnd"]:
        if col in visits.columns:
            visits[col] = visits[col].apply(time_to_minutes)

    # Drop non-essential columns that are strings and not directly used numerically
    carer_cols_to_drop = ["Date", "Status", "PreferredShift", "WorkingDays", "OffDays"]
    for col in carer_cols_to_drop:
        if col in carers.columns:
            carers = carers.drop(columns=[col])

    visit_cols_to_drop = ["PatientID"]
    for col in visit_cols_to_drop:
        if col in visits.columns:
            visits = visits.drop(columns=[col])

    # Ensure all numerical columns are indeed numeric, coercing errors to NaN
    for df in [carers, visits]:
        for col in df.columns:
            # Exclude ID columns and set-like columns from general numeric conversion
            if df[col].dtype == 'object' and col not in ['CarerID', 'VisitID', 'Skills', 'Languages', 'RequiredSkills', 'PreferredLanguage']:
                df[col] = pd.to_numeric(df[col], errors='coerce')

    return carers, visits
'''

# Write the content to the file
with open(os.path.join(PROJECT_ROOT, "utils", "preprocessing.py"), "w") as f:
    f.write(preprocessing_code)

# Define the content for config/settings.py
settings_code = '''
# Reward Function Weights
HARD_CONSTRAINT_REWARD = 100
HARD_CONSTRAINT_PENALTY = -100
WORKLOAD_BALANCE_WEIGHT = 30
PRIORITY_WEIGHT = 10
SKILL_MATCH_REWARD = 5
LANGUAGE_MATCH_REWARD = 5
DISTANCE_WEIGHT = -0.1 # Added missing definition for DISTANCE_WEIGHT
'''

# Write the content to the file
with open(os.path.join(PROJECT_ROOT, "config", "settings.py"), "w") as f:
    f.write(settings_code)

VanillaDQN Project Created Successfully!
VanillaDQN/
    requirements.txt
    README.md
    saved_models/
    rewards/
        __init__.py
        reward_function.py
        __pycache__/
            reward_function.cpython-312.pyc
            __init__.cpython-312.pyc
    config/
        settings.py
        __init__.py
        __pycache__/
            settings.cpython-312.pyc
            __init__.cpython-312.pyc
    notebooks/
    agents/
        dqn_agent.py
        replay_buffer.py
        dqn_network.py
        __init__.py
        __pycache__/
            dqn_agent.cpython-312.pyc
            replay_buffer.cpython-312.pyc
            __init__.cpython-312.pyc
            dqn_network.cpython-312.pyc
    results/
    utils/
        helpers.py
        distance.py
        __init__.py
        preprocessing.py
        __pycache__/
            __init__.cpython-312.pyc
            preprocessing.cpython-312.pyc
            distance.cpython-312.pyc
    data/
        carers.csv
        distance_

In [4]:
import pandas as pd

carers = pd.read_csv("/content/drive/MyDrive/VanillaDQN/data/carers_available.csv")

print(carers.columns.tolist())

['CarerID', 'WorkerType', 'Latitude', 'Longitude', 'PreferredAreaLatitude', 'PreferredAreaLongitude', 'MaxTravelDistanceMiles', 'ExperienceInYears', 'Skills', 'Gender', 'Languages', 'VehicleType', 'WorkingDays', 'OffDays', 'ShiftStart', 'ShiftEnd', 'ContractHoursPerWeek', 'MaxDailyHours', 'MaxWeeklyHours', 'PreferredShift', 'Date', 'DayOfWeek', 'Status']


In [5]:
visits = pd.read_csv("/content/drive/MyDrive/VanillaDQN/data/visits.csv")

print(visits.columns.tolist())

['VisitID', 'PatientID', 'Day', 'Latitude', 'Longitude', 'VisitDurationMinutes', 'TimeWindowStart', 'TimeWindowEnd', 'PriorityLevel', 'ComplexityLevel', 'RequiredSkills', 'PreferredGender', 'PreferredLanguage', 'TwoCarerRequired', 'ContinuityRequirement']


In [7]:
import os

data_path = "/content/drive/MyDrive/VanillaDQN/data"

print(os.listdir(data_path))

['carers_available.csv', 'visits.csv']


In [8]:
from utils.preprocessing import preprocess_data

carers_processed, visits_processed = preprocess_data()

print(carers_processed.shape)
print(visits_processed.shape)

(462, 23)
(3118, 15)


In [9]:
import sys

PROJECT_ROOT = "/content/drive/MyDrive/VanillaDQN"

if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)


from utils.preprocessing import preprocess_data


carers_processed, visits_processed = preprocess_data()


print("=" * 60)
print("Preprocessing Successful")
print("=" * 60)


print("\nCarers Shape:")
print(carers_processed.shape)


print("\nVisits Shape:")
print(visits_processed.shape)


print("\nCarer Columns:")
print(carers_processed.columns.tolist())


print("\nVisit Columns:")
print(visits_processed.columns.tolist())


print("\nFirst Carer Skills:")
print(carers_processed.iloc[0]["Skills"])


print("\nFirst Carer Languages:")
print(carers_processed.iloc[0]["Languages"])


print("\nFirst Visit Required Skills:")
print(visits_processed.iloc[0]["RequiredSkills"])


print("\nFirst Visit Preferred Language:")
print(visits_processed.iloc[0]["PreferredLanguage"])

Preprocessing Successful

Carers Shape:
(462, 23)

Visits Shape:
(3118, 15)

Carer Columns:
['CarerID', 'WorkerType', 'Latitude', 'Longitude', 'PreferredAreaLatitude', 'PreferredAreaLongitude', 'MaxTravelDistanceMiles', 'ExperienceInYears', 'Skills', 'Gender', 'Languages', 'VehicleType', 'WorkingDays', 'OffDays', 'ShiftStart', 'ShiftEnd', 'ContractHoursPerWeek', 'MaxDailyHours', 'MaxWeeklyHours', 'PreferredShift', 'Date', 'DayOfWeek', 'Status']

Visit Columns:
['VisitID', 'PatientID', 'Day', 'Latitude', 'Longitude', 'VisitDurationMinutes', 'TimeWindowStart', 'TimeWindowEnd', 'PriorityLevel', 'ComplexityLevel', 'RequiredSkills', 'PreferredGender', 'PreferredLanguage', 'TwoCarerRequired', 'ContinuityRequirement']

First Carer Skills:
{'Palliative Care', 'Medication Support'}

First Carer Languages:
{'French', 'English'}

First Visit Required Skills:
{'Personal Care', 'Companionship'}

First Visit Preferred Language:
{'Arabic'}


In [10]:
import sys

PROJECT_ROOT = "/content/drive/MyDrive/VanillaDQN"

if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)


from utils.preprocessing import preprocess_data

from utils.distance import (
    create_distance_matrix,
    save_distance_matrix,
    load_distance_matrix
)


carers_processed, visits_processed = preprocess_data()


distance_matrix = create_distance_matrix(

    carers_processed,

    visits_processed

)


print("Distance Matrix Shape:")
print(distance_matrix.shape)


print("\nFirst 5 distances:")
print(distance_matrix[0][:5])


save_distance_matrix(distance_matrix)


loaded_matrix = load_distance_matrix()


print("\nLoaded Matrix Shape:")
print(loaded_matrix.shape)

Distance Matrix Shape:
(462, 3118)

First 5 distances:
[ 3.72097831 16.54290691 16.54290691 16.54290691 16.54290691]

Loaded Matrix Shape:
(462, 3118)


In [11]:
from constraints.constraint_engine import ConstraintEngine


constraint_engine = ConstraintEngine(

    carers_processed,

    visits_processed,

    distance_matrix

)


mask = constraint_engine.get_action_mask(0)


print("Total actions:")
print(len(mask))


print("\nFeasible actions:")
print(mask.sum())


print("\nFirst 20:")
print(mask[:20])

Total actions:
462

Feasible actions:
30.0

First 20:
[0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0.]


In [18]:
import environment.healthcare_env as env_file

print(env_file.__file__)

/content/drive/MyDrive/VanillaDQN/environment/healthcare_env.py


In [19]:
import importlib
import environment.healthcare_env

importlib.reload(environment.healthcare_env)

<module 'environment.healthcare_env' from '/content/drive/MyDrive/VanillaDQN/environment/healthcare_env.py'>

In [20]:
from environment.healthcare_env import HealthcareEnv


env = HealthcareEnv(

    carers_processed,

    visits_processed,

    constraint_engine,

    reward_function

)

In [21]:
print(
    visits_processed.iloc[0]["PriorityLevel"]
)

print(
    type(
        visits_processed.iloc[0]["PriorityLevel"]
    )
)

Low
<class 'str'>


In [22]:
from environment.healthcare_env import HealthcareEnv

In [23]:
env = HealthcareEnv(

    carers_processed,

    visits_processed,

    constraint_engine,

    reward_function

)


state = env.reset()


print(state)

[51.36008  -0.273553 45.        1.        0.      ]


In [24]:
import torch

from agents.dqn_network import DQNNetwork



state_size = 5

action_size = len(

    constraint_engine.get_action_mask(0)

)



network = DQNNetwork(

    state_size,

    action_size

)



dummy_state = torch.FloatTensor(

    state

).unsqueeze(0)



q_values = network(

    dummy_state

)



print("Q-value shape:")

print(q_values.shape)

Q-value shape:
torch.Size([1, 462])


In [25]:
from agents.dqn_agent import DQNAgent


action_size = len(

    constraint_engine.get_action_mask(0)

)


agent = DQNAgent(

    state_size=5,

    action_size=action_size

)


mask = env.get_action_mask()


action = agent.choose_action(

    state,

    mask

)


print("Selected action:")

print(action)

Selected action:
321


In [26]:
from agents.replay_buffer import ReplayBuffer


memory = ReplayBuffer(

    capacity=100

)


memory.push(

    state,

    10,

    50,

    state,

    False

)


print("Buffer size:")

print(len(memory))


print(memory.sample(1))

Buffer size:
1
[(array([51.36008 , -0.273553, 45.      ,  1.      ,  0.      ]), 10, 50, array([51.36008 , -0.273553, 45.      ,  1.      ,  0.      ]), False)]


In [28]:
import agents.dqn_agent as dqn_file

print(dqn_file.__file__)

/content/drive/MyDrive/VanillaDQN/agents/dqn_agent.py


In [29]:
import importlib
import agents.dqn_agent

importlib.reload(agents.dqn_agent)

<module 'agents.dqn_agent' from '/content/drive/MyDrive/VanillaDQN/agents/dqn_agent.py'>

In [30]:
from agents.dqn_agent import DQNAgent

In [31]:
agent = DQNAgent(

    state_size=5,

    action_size=462

)

In [32]:
print(
    hasattr(agent, "learn")
)

True


In [33]:
loss = agent.learn(batch)

print("Loss:")
print(loss)

Loss:
3527.222412109375


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/loss.py:626: UserWarning: Using a target size (torch.Size([1])) that is different to the input size (torch.Size([])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


In [34]:
import importlib
import agents.dqn_agent

importlib.reload(agents.dqn_agent)

from agents.dqn_agent import DQNAgent

In [35]:
agent = DQNAgent(

    state_size=5,

    action_size=462

)


loss = agent.learn(batch)


print("Loss:")
print(loss)

Loss:
3193.950439453125


In [37]:
# ==========================================================
# Step 12.1 — Import Training Module
# ==========================================================

import importlib
import training.train

importlib.reload(training.train)

from training.train import train_dqn

print("Training module imported successfully.")

Training module imported successfully.


In [38]:
# ==========================================================
# Step 12.2 — Create Replay Buffer
# ==========================================================

from agents.replay_buffer import ReplayBuffer

memory = ReplayBuffer(capacity=10000)

print("Replay buffer created.")

Replay buffer created.


In [39]:
# ==========================================================
# Step 12.3 — Reset Reward Workload State
# ==========================================================

import numpy as np

reward_function.visit_counts = np.zeros(
    len(carers_processed),
    dtype=float
)

print("Reward workload counters reset.")

Reward workload counters reset.


In [42]:
import config.settings as settings

print(settings.__file__)

print(hasattr(settings, "DISTANCE_WEIGHT"))

/content/drive/MyDrive/VanillaDQN/config/settings.py
False


In [43]:
import config.settings as settings

for name in dir(settings):
    if "WEIGHT" in name or "REWARD" in name or "PENALTY" in name:
        print(name, "=", getattr(settings, name))

HARD_CONSTRAINT_PENALTY = -100
HARD_CONSTRAINT_REWARD = 100
LANGUAGE_MATCH_REWARD = 5
PRIORITY_WEIGHT = 10
SKILL_MATCH_REWARD = 5
WORKLOAD_BALANCE_WEIGHT = 30


In [47]:
import importlib

import config.settings
import rewards.reward_function
import environment.healthcare_env
import agents.dqn_agent
import training.train

importlib.reload(config.settings)
importlib.reload(rewards.reward_function)
importlib.reload(environment.healthcare_env)
importlib.reload(agents.dqn_agent)
importlib.reload(training.train)

print("Modules reloaded.")

Modules reloaded.


In [48]:
from rewards.reward_function import RewardFunction
from environment.healthcare_env import HealthcareEnv
from agents.dqn_agent import DQNAgent
from agents.replay_buffer import ReplayBuffer

reward_function = RewardFunction(
    carers_processed,
    visits_processed,
    constraint_engine
)

env = HealthcareEnv(
    carers_processed,
    visits_processed,
    constraint_engine,
    reward_function
)

agent = DQNAgent(
    state_size=5,
    action_size=len(carers_processed)
)

memory = ReplayBuffer(capacity=10000)

print("Objects recreated successfully.")

Objects recreated successfully.


In [49]:
import sys

PROJECT_ROOT = "/content/drive/MyDrive/VanillaDQN"

if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)


from config.settings import (
    HARD_CONSTRAINT_REWARD,
    HARD_CONSTRAINT_PENALTY,
    WORKLOAD_BALANCE_WEIGHT,
    PRIORITY_WEIGHT,
    SKILL_MATCH_REWARD,
    LANGUAGE_MATCH_REWARD,
    DISTANCE_WEIGHT,
)


print("Hard reward:", HARD_CONSTRAINT_REWARD)
print("Hard penalty:", HARD_CONSTRAINT_PENALTY)
print("Workload weight:", WORKLOAD_BALANCE_WEIGHT)
print("Priority weight:", PRIORITY_WEIGHT)
print("Skill weight:", SKILL_MATCH_REWARD)
print("Language weight:", LANGUAGE_MATCH_REWARD)
print("Distance weight:", DISTANCE_WEIGHT)

Hard reward: 100
Hard penalty: -100
Workload weight: 30
Priority weight: 10
Skill weight: 5
Language weight: 5
Distance weight: -0.1


In [50]:
from rewards.reward_function import RewardFunction
from environment.healthcare_env import HealthcareEnv


reward_function = RewardFunction(
    carers_processed,
    visits_processed,
    constraint_engine,
)


env = HealthcareEnv(
    carers_processed,
    visits_processed,
    constraint_engine,
    reward_function,
)


print("Reward function and environment recreated successfully.")

Reward function and environment recreated successfully.


In [51]:
import numpy as np


state = env.reset()

mask = env.get_action_mask()

feasible_actions = np.flatnonzero(mask == 1)

carer_index = int(feasible_actions[0])


components = reward_function.calculate_reward_components(
    carer_index=carer_index,
    visit_index=0,
    update_workload=False,
)


print(components)

{'feasible': True, 'hard': 100.0, 'workload': -0.0, 'priority': 10.0, 'skill': 0.0, 'language': 0.0, 'distance': -0.06779379904362277, 'total': 109.93220620095637}


In [52]:
from training.train import train_dqn

rewards = train_dqn(
    env=env,
    agent=agent,
    memory=memory,
    episodes=1,
    batch_size=8
)

print("\nReward history:")
print(rewards)

print("\nReplay buffer size:")
print(len(memory))

print("\nFinal epsilon:")
print(agent.epsilon)

Episode: 1
Total reward: 295056.81828900805
Epsilon: 0.995

Reward history:
[295056.81828900805]

Replay buffer size:
3118

Final epsilon:
0.995


In [54]:
import importlib
import config.settings

importlib.reload(config.settings)

from config.settings import (
    CARERS_FILE,
    VISITS_FILE,
    DISTANCE_MATRIX_FILE,
    MODEL_FILE,
    TRAINING_RESULTS_FILE,
    EVALUATION_RESULTS_FILE,
    ASSIGNMENTS_FILE,
    WORKLOAD_RESULTS_FILE,
    DISTANCE_WEIGHT,
    WORKLOAD_BALANCE_WEIGHT,
    STATE_SIZE,
)


print("Carers file:", CARERS_FILE)
print("Visits file:", VISITS_FILE)
print("Distance matrix:", DISTANCE_MATRIX_FILE)
print("Model file:", MODEL_FILE)
print("Training results:", TRAINING_RESULTS_FILE)
print("Evaluation results:", EVALUATION_RESULTS_FILE)
print("Assignments:", ASSIGNMENTS_FILE)
print("Workloads:", WORKLOAD_RESULTS_FILE)

print("\nState size:", STATE_SIZE)
print("Workload weight:", WORKLOAD_BALANCE_WEIGHT)
print("Distance weight:", DISTANCE_WEIGHT)

Carers file: /content/drive/MyDrive/VanillaDQN/data/carers.csv
Visits file: /content/drive/MyDrive/VanillaDQN/data/visits.csv
Distance matrix: /content/drive/MyDrive/VanillaDQN/data/distance_matrix.npy
Model file: /content/drive/MyDrive/VanillaDQN/saved_models/vanilla_dqn.pth
Training results: /content/drive/MyDrive/VanillaDQN/results/training_results.csv
Evaluation results: /content/drive/MyDrive/VanillaDQN/results/evaluation_results.csv
Assignments: /content/drive/MyDrive/VanillaDQN/results/visit_assignments.csv
Workloads: /content/drive/MyDrive/VanillaDQN/results/carer_workloads.csv

State size: 5
Workload weight: 30.0
Distance weight: 10.0


In [55]:
import importlib
import training.train

importlib.reload(training.train)

from training.train import train_dqn

print("Training module imported successfully.")

Training module imported successfully.


In [56]:
import importlib
import agents.dqn_network
import agents.dqn_agent

importlib.reload(agents.dqn_network)
importlib.reload(agents.dqn_agent)

from agents.dqn_network import DQNNetwork
from agents.dqn_agent import DQNAgent

print("Final DQN network and agent loaded successfully.")

Final DQN network and agent loaded successfully.


In [57]:
import numpy as np
import torch

from config.settings import STATE_SIZE


action_size = len(carers_processed)

agent = DQNAgent(
    state_size=STATE_SIZE,
    action_size=action_size,
)


state = env.reset()

mask = env.get_action_mask()


action = agent.choose_action(
    state=state,
    action_mask=mask,
    training=True,
)


dummy_state = torch.as_tensor(
    state,
    dtype=torch.float32,
).unsqueeze(0).to(agent.device)


with torch.no_grad():
    q_values = agent.network(dummy_state)


print("Device:", agent.device)
print("State size:", agent.state_size)
print("Action size:", agent.action_size)
print("Q-value shape:", q_values.shape)
print("Feasible actions:", int(mask.sum()))
print("Selected feasible action:", action)
print("Selected action mask value:", mask[action])
print("Target network ready:", agent.target_network is not None)

Device: cpu
State size: 5
Action size: 462
Q-value shape: torch.Size([1, 462])
Feasible actions: 30
Selected feasible action: 5
Selected action mask value: 1.0
Target network ready: True


In [58]:
import importlib
import agents.replay_buffer
import training.train

importlib.reload(agents.replay_buffer)
importlib.reload(training.train)

from agents.replay_buffer import ReplayBuffer
from training.train import train_dqn

print("Final replay buffer and trainer loaded successfully.")

Final replay buffer and trainer loaded successfully.


In [59]:
from agents.dqn_agent import DQNAgent
from config.settings import (
    BUFFER_SIZE,
    STATE_SIZE,
)


agent = DQNAgent(
    state_size=STATE_SIZE,
    action_size=len(carers_processed),
)


memory = ReplayBuffer(
    capacity=BUFFER_SIZE
)


print("Fresh agent and replay buffer created.")

Fresh agent and replay buffer created.


In [60]:
rewards = train_dqn(
    env=env,
    agent=agent,
    memory=memory,
    episodes=1,
    batch_size=64,
    target_update_frequency=10,
    save_model_every=10,
    print_every=1,
    max_steps_per_episode=None,
    save_results=True,
    save_checkpoints=False,
)


print("Reward history:")
print(rewards)

Episode: 1
Total reward: 301825.0276
Average loss: 24.008153
Epsilon: 0.995000
Coverage: 1.0000
Feasibility rate: 1.0000
Average travel distance: 16.5432 miles
Distance compliance rate: 0.1899
Workload variance: 21.1274
Workload standard deviation: 4.5965
Fairness index: 0.6831
Replay buffer size: 3118
Episode time: 192.12 seconds

Reward history:
[301825.02760947566]


In [61]:
training=False

In [62]:
import os
import pandas as pd

from config.settings import TRAINING_RESULTS_FILE

# Read the training results
training_results = pd.read_csv(TRAINING_RESULTS_FILE)

print(training_results)

print("\nSaved at:")
print(TRAINING_RESULTS_FILE)

   episode   total_reward  average_loss  epsilon  processed_visits  \
0        1  301825.027609     24.008153    0.995              3118   

   total_visits  coverage  feasible_assignments  infeasible_assignments  \
0          3118       1.0                  3118                       0   

   feasibility_rate  average_travel_distance  within_distance_limit  \
0               1.0                16.543182                    592   

   outside_distance_limit  distance_compliance_rate  workload_variance  \
0                    2526                  0.189865          21.127434   

   workload_std  fairness_index  episode_time_seconds  replay_buffer_size  \
0      4.596459         0.68313            192.123734                3118   

   training_steps  
0            3055  

Saved at:
/content/drive/MyDrive/VanillaDQN/results/training_results.csv


In [63]:
import importlib
import evaluation.metrics
import evaluation.evaluate

importlib.reload(evaluation.metrics)
importlib.reload(evaluation.evaluate)

from evaluation.evaluate import evaluate_dqn

print("Evaluation modules loaded successfully.")

Evaluation modules loaded successfully.


In [64]:
evaluation_results, assignments, workloads = evaluate_dqn(
    env=env,
    agent=agent,
    save_results=True,
)


print("Evaluation results:")
display(evaluation_results)


print("\nFirst 10 assignments:")
display(assignments.head(10))


print("\nFirst 10 workload records:")
display(workloads.head(10))

Evaluation results:


,processed_visits,total_visits,coverage,feasible_assignments,infeasible_assignments,feasibility_rate,total_travel_distance,average_travel_distance,within_distance_limit,outside_distance_limit,distance_compliance_rate,total_reward,average_reward,workload_variance,workload_std,fairness_index,active_carer_nodes,computation_time_seconds,epsilon_used,policy_mode
0,3118,3118,1.0,3118,0,1.0,51532.347186,16.527372,510,2608,0.163566,-3.910327e+06,-1254.113907,629.066828,25.081205,0.067517,69,175.58165,0.0,greedy



First 10 assignments:


,VisitIndex,VisitID,PatientID,VisitDay,PriorityLevel,VisitDurationMinutes,TimeWindowStart,TimeWindowEnd,CarerActionIndex,CarerID,CarerDayOfWeek,Feasible,TravelDistanceMiles,MaximumDistanceMiles,WithinDistanceLimit,Reward
0,0,V000001,P0001,Tuesday,Low,45,900,1080,321,C0073,Tuesday,True,6.589916,12.0,True,112.427458
1,1,V000002,P0002,Tuesday,Medium,45,600,780,105,C0024,Tuesday,True,16.627615,5.0,False,120.297487
2,2,V000003,P0002,Wednesday,Medium,45,600,780,458,C0100,Wednesday,True,16.457135,5.0,False,120.359013
3,3,V000004,P0002,Saturday,Medium,45,600,780,145,C0035,Saturday,True,11.751224,12.0,True,120.143769
4,4,V000005,P0002,Monday,Medium,45,600,780,130,C0032,Monday,True,19.061404,5.0,False,125.540968
5,5,V000006,P0002,Friday,Medium,45,600,780,134,C0032,Friday,True,19.061404,5.0,False,125.605903
6,6,V000007,P0002,Sunday,Medium,45,600,780,16,C0004,Sunday,True,18.206646,12.0,False,120.441332
7,7,V000008,P0002,Thursday,Medium,45,600,780,400,C0088,Thursday,True,13.390247,12.0,False,120.466131
8,8,V000009,P0003,Friday,Medium,60,1020,1200,249,C0058,Friday,True,18.521932,12.0,False,125.573830
9,9,V000010,P0003,Thursday,Medium,60,1020,1200,267,C0062,Thursday,True,19.547823,5.0,False,120.875372



First 10 workload records:


,CarerActionIndex,CarerID,DayOfWeek,AssignedVisitCount
0,0,C0001,Monday,0.0
1,1,C0001,Wednesday,0.0
2,2,C0001,Friday,0.0
3,3,C0001,Saturday,0.0
4,4,C0002,Monday,0.0
5,5,C0002,Tuesday,0.0
6,6,C0002,Wednesday,111.0
7,7,C0002,Friday,0.0
8,8,C0002,Saturday,0.0
9,9,C0003,Monday,0.0


In [65]:
import os

from config.settings import (
    ASSIGNMENTS_FILE,
    EVALUATION_RESULTS_FILE,
    WORKLOAD_RESULTS_FILE,
)


print(
    "Evaluation CSV:",
    os.path.exists(EVALUATION_RESULTS_FILE),
)

print(
    "Assignments CSV:",
    os.path.exists(ASSIGNMENTS_FILE),
)

print(
    "Workloads CSV:",
    os.path.exists(WORKLOAD_RESULTS_FILE),
)

Evaluation CSV: True
Assignments CSV: True
Workloads CSV: True


In [66]:
from google.colab import files

from config.settings import (
    ASSIGNMENTS_FILE,
    EVALUATION_RESULTS_FILE,
    WORKLOAD_RESULTS_FILE,
)


files.download(EVALUATION_RESULTS_FILE)
files.download(ASSIGNMENTS_FILE)
files.download(WORKLOAD_RESULTS_FILE)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>